In [13]:
!apt-get install -qq tesseract-ocr tesseract-ocr-ara poppler-utils
!pip install -q pytesseract pdf2image transformers torch scikit-learn xgboost joblib

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
BOOK_PATH = "/content/drive/MyDrive/zaid t/_الحيوان5.pdf"

In [15]:
import re, json, warnings
import numpy as np
import pandas as pd
import torch
import joblib
import pytesseract
from pdf2image import convert_from_path
from pdf2image.pdf2image import pdfinfo_from_path
from multiprocessing import Pool
from collections import Counter
from transformers import AutoTokenizer, AutoModelForSequenceClassification

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

LABEL_TO_CATEGORY = {0: '٣-٨', 1: '٩-١٢', 2: '١٣-١٨', 3: '١٨-٢١', 4: '٢١+'}
NUM_LABELS = 5
MIN_CHUNK_LENGTH = 50
FEATURE_COLS = ['total_words', 'total_chars', 'num_chunks', 'num_sentences', 'avg_word_length', 'std_word_length', 'max_word_length', 'min_word_length', 'median_word_length', 'avg_sentence_length', 'std_sentence_length', 'max_sentence_length', 'words_per_sentence', 'vocab_size', 'type_token_ratio', 'hapax_ratio', 'long_word_ratio', 'short_word_ratio', 'medium_word_ratio', 'punctuation_density', 'question_mark_ratio', 'exclamation_ratio', 'dialogue_ratio', 'avg_chunk_words', 'std_chunk_words', 'reading_ease_proxy', 'freq_mean', 'freq_std', 'freq_max', 'freq_skewness', 'space_ratio', 'bigram_ratio']

def ocr_page(args):
    pdf_path, page_num, dpi = args
    try:
        imgs = convert_from_path(pdf_path, first_page=page_num, last_page=page_num, dpi=dpi)
        text = pytesseract.image_to_string(imgs[0], lang='ara').strip()
        del imgs
        return (page_num, text)
    except:
        return (page_num, '')

def extract_text(pdf_path, dpi=300, workers=4):
    total = pdfinfo_from_path(pdf_path)['Pages']
    tasks = [(pdf_path, p, dpi) for p in range(1, total + 1)]
    results = {}
    with Pool(workers) as pool:
        for page_num, text in pool.imap_unordered(ocr_page, tasks):
            results[page_num] = text
            print(f'\r{len(results)}/{total}', end='', flush=True)
    print()
    return '\n'.join(results[p] for p in sorted(results)), total

def clean(text):
    if not isinstance(text, str): return ''
    text = re.sub(r'[^\u0600-\u06FF\u0750-\u077F\uFB50-\uFDFF\uFE70-\uFEFF\s\.\,\;\:\!\?\-\(\)\"\'«»]', ' ', text)
    text = re.sub(r'[إأآا]', 'ا', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'[\u064B-\u065F]', '', text)
    return re.sub(r'\s+', ' ', text).strip()

def make_chunks(text, tokenizer, size=512):
    ids = tokenizer.encode(text, add_special_tokens=False)
    chunks = []
    for i in range(0, len(ids), size):
        d = tokenizer.decode(ids[i:i+size], skip_special_tokens=True).strip()
        if d: chunks.append(d)
    return chunks

def get_features(df):
    rows = []
    for book, group in df.groupby('Book Name'):
        texts = group['Clean_Content'].tolist()
        full  = ' '.join(texts)
        words = full.split()
        n     = max(len(words), 1)
        wl    = [len(w) for w in words] if words else [0]
        sents = [s.strip() for s in re.split(r'[.!?؟،\n]+', full) if len(s.strip()) > 5]
        ns    = max(len(sents), 1)
        sl    = [len(s.split()) for s in sents] if sents else [0]
        wf    = Counter(words)
        fv    = list(wf.values()) if wf else [0]
        bg    = [' '.join(words[i:i+2]) for i in range(len(words)-1)]
        awl   = np.mean(wl)
        asl   = np.mean(sl)
        rows.append({
            'Book Name': book, 'label': int(group['label'].iloc[0]),
            'total_words': n, 'total_chars': len(full),
            'num_chunks': len(texts), 'num_sentences': ns,
            'avg_word_length': awl, 'std_word_length': np.std(wl),
            'max_word_length': max(wl), 'min_word_length': min(wl),
            'median_word_length': np.median(wl),
            'avg_sentence_length': asl, 'std_sentence_length': np.std(sl),
            'max_sentence_length': max(sl), 'words_per_sentence': n / ns,
            'vocab_size': len(set(words)), 'type_token_ratio': len(set(words)) / n,
            'hapax_ratio': sum(1 for w,c in wf.items() if c==1) / n,
            'long_word_ratio': len([w for w in words if len(w)>6]) / n,
            'short_word_ratio': len([w for w in words if len(w)<3]) / n,
            'medium_word_ratio': len([w for w in words if 3<=len(w)<=6]) / n,
            'punctuation_density': (full.count('.')+full.count('،')+full.count(',')+full.count('؟')+full.count('?')+full.count('!')) / n,
            'question_mark_ratio': (full.count('؟')+full.count('?')) / ns,
            'exclamation_ratio': full.count('!') / ns,
            'dialogue_ratio': (full.count(':')+full.count('«')+full.count('»')+full.count('"')) / ns,
            'avg_chunk_words': np.mean([len(t.split()) for t in texts]),
            'std_chunk_words': np.std([len(t.split()) for t in texts]),
            'reading_ease_proxy': 206.835 - 1.015*asl - 84.6*(awl/2.5),
            'freq_mean': np.mean(fv), 'freq_std': np.std(fv),
            'freq_max': max(fv),
            'freq_skewness': (3*(np.mean(fv)-np.median(fv)))/np.std(fv) if np.std(fv)>0 else 0,
            'space_ratio': full.count(' ') / max(len(full),1),
            'bigram_ratio': len(set(bg)) / max(len(bg),1),
        })
    return pd.DataFrame(rows)

def predict(chunks, model, tokenizer, ml_models, scaler, feature_cols, w_llm, w_ml):
    clean_chunks = [clean(c) for c in chunks]
    clean_chunks = [c for c in clean_chunks if len(c) >= MIN_CHUNK_LENGTH]

    all_probs, preds = [], []
    model.eval()
    with torch.no_grad():
        for c in clean_chunks:
            enc   = tokenizer(c, max_length=512, padding='max_length', truncation=True, return_tensors='pt')
            enc   = {k: v.to(device) for k, v in enc.items()}
            probs = torch.softmax(model(**enc).logits, dim=-1).cpu().numpy()[0]
            all_probs.append(probs)
            preds.append(int(np.argmax(probs)))

    votes       = Counter(preds)
    llm_pred    = votes.most_common(1)[0][0]
    llm_conf    = votes.most_common(1)[0][1] / len(preds)
    llm_probs   = np.mean(all_probs, axis=0)

    tmp = pd.DataFrame({'Book Name': ['b']*len(clean_chunks), 'Clean_Content': clean_chunks, 'label': [0]*len(clean_chunks)})
    feats = get_features(tmp)
    X = scaler.transform(feats[feature_cols].values.astype(float))
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    ml_probs = np.mean([m.predict_proba(X) for m in ml_models.values()], axis=0)[0]

    final = w_llm * llm_probs + w_ml * ml_probs
    pred  = int(np.argmax(final))

    return {
        'predicted': LABEL_TO_CATEGORY[pred],
        'confidence': float(final[pred]),
        'arabert': LABEL_TO_CATEGORY[llm_pred],
        'arabert_conf': float(llm_conf),
        'ml': LABEL_TO_CATEGORY[int(np.argmax(ml_probs))],
        'ml_conf': float(ml_probs.max()),
        'probs': {LABEL_TO_CATEGORY[i]: float(final[i]) for i in range(NUM_LABELS)},
        'chunks': len(clean_chunks),
        'votes': {LABEL_TO_CATEGORY[k]: v for k, v in votes.items()},
    }

print('ready')

ready


In [16]:
feature_cols = ['total_words', 'total_chars', 'num_chunks', 'num_sentences',
    'avg_word_length', 'std_word_length', 'max_word_length', 'min_word_length',
    'median_word_length', 'avg_sentence_length', 'std_sentence_length',
    'max_sentence_length', 'words_per_sentence', 'vocab_size', 'type_token_ratio',
    'hapax_ratio', 'long_word_ratio', 'short_word_ratio', 'medium_word_ratio',
    'punctuation_density', 'question_mark_ratio', 'exclamation_ratio',
    'dialogue_ratio', 'avg_chunk_words', 'std_chunk_words', 'reading_ease_proxy',
    'freq_mean', 'freq_std', 'freq_max', 'freq_skewness', 'space_ratio', 'bigram_ratio']

w_llm = 0.5
w_ml  = 0.5

tokenizer = AutoTokenizer.from_pretrained('/content/drive/MyDrive/zaid t/arabert_finetuned')
model = AutoModelForSequenceClassification.from_pretrained(
    '/content/drive/MyDrive/zaid t/arabert_finetuned', num_labels=NUM_LABELS, ignore_mismatched_sizes=True
).to(device)

ml_models = {
    'xgb': joblib.load('/content/drive/MyDrive/zaid t/xgb_model.pkl'),
    'rf':  joblib.load('/content/drive/MyDrive/zaid t/rf_model.pkl'),
    'gb':  joblib.load('/content/drive/MyDrive/zaid t/gb_model.pkl'),
}
scaler = joblib.load('/content/drive/MyDrive/zaid t/scaler.pkl')
print('models loaded')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

models loaded


In [17]:
text, pages = extract_text(BOOK_PATH)
chunks = make_chunks(text, tokenizer)
result = predict(chunks, model, tokenizer, ml_models, scaler, feature_cols, w_llm, w_ml)

print(f"pages:      {pages}")
print(f"chunks:     {result['chunks']}")
print(f"prediction: {result['predicted']}")
print(f"confidence: {result['confidence']:.1%}")
print(f"arabert:    {result['arabert']} ({result['arabert_conf']:.1%})")
print(f"ml:         {result['ml']} ({result['ml_conf']:.1%})")
print(f"probs:      {result['probs']}")
print(f"votes:      {result['votes']}")

639/639


Token indices sequence length is longer than the specified maximum sequence length for this model (300034 > 512). Running this sequence through the model will result in indexing errors


pages:      639
chunks:     586
prediction: ١٨-٢١
confidence: 86.2%
arabert:    ١٨-٢١ (98.8%)
ml:         ١٨-٢١ (73.7%)
probs:      {'٣-٨': 0.001240853223931856, '٩-١٢': 0.021370565441334477, '١٣-١٨': 0.001649683370590414, '١٨-٢١': 0.8623136609756934, '٢١+': 0.11342601023265995}
votes:      {'١٨-٢١': 579, '٢١+': 7}
